<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# APIs Minilab Crimes - Solution

---

# Scenario
You want to find out 2 things:
1. Which crimes are most common around Piccadilly Circus (in London, UK)?
2. If your bike is stolen in Oxford, how likely are you to get it back?

First of all though you'll go through a guided walk through to show you how to transform data from APIs into a `pandas` `DataFrame`.

In [1]:
import pandas as pd
import requests

---

# Getting and analysing police data
    
Let's apply our `pandas` skills to our police data problem!
    
Let's start off by making a police API request again from this API: https://data.police.uk/docs/

In [2]:
lat= str(51.508624722468454)
lon= str(-0.14708795945249462)
date='2023-10'

police_url = 'https://data.police.uk/api/crimes-street/all-crime?lat='+lat+'&lng='+lon+'&date='+date
police_json = requests.get(police_url).json()

We can easily convert JSON data straight into a `DataFrame` using the `pd.DataFrame()` command

In [3]:
crimes = pd.DataFrame(police_json)

What do you notice about the DataFrame when we preview it? Something doesn't look quite right...

In [4]:
crimes.head()

,category,location_type,location,context,outcome_status,persistent_id,id,location_subtype,month
0,anti-social-behaviour,Force,"{'latitude': '51.498404', 'street': {'id': 167...",,None,,113798885,,2023-10
1,anti-social-behaviour,Force,"{'latitude': '51.517476', 'street': {'id': 167...",,None,,113798685,,2023-10
2,anti-social-behaviour,Force,"{'latitude': '51.511783', 'street': {'id': 167...",,None,,113798656,,2023-10
3,anti-social-behaviour,Force,"{'latitude': '51.507576', 'street': {'id': 167...",,None,,113798553,,2023-10
4,anti-social-behaviour,Force,"{'latitude': '51.516668', 'street': {'id': 167...",,None,,113795992,,2023-10


It looks like the `location` column hasn't been expanded out correctly, because the JSON in that field has a lot of nesting.

We can fix this issue using the `json_normalize` function. This expands the column out fully.

In [5]:
pd.json_normalize(crimes['location'])

,latitude,longitude,street.id,street.name
0,51.498404,-0.160567,1673888,On or near Herbert Crescent
1,51.517476,-0.136579,1677079,On or near Supermarket
2,51.511783,-0.129778,1678199,On or near Nightclub
3,51.507576,-0.141776,1676403,On or near Nightclub
4,51.516668,-0.141210,1676419,On or near Margaret Street
...,...,...,...,...
6321,51.496440,-0.135867,1677223,On or near Spencer Place
6322,51.512247,-0.130120,1678100,On or near Gerrard Place
6323,51.499149,-0.164514,1673361,On or near Brompton Road
6324,51.519427,-0.132074,1677850,On or near South Crescent


And we can join this together with our original DataFrame using the `concat` method.

In [6]:
crimes = pd.concat([
    crimes,
    pd.json_normalize(crimes['location'])
    ],
    axis=1)

After concatenating, we can see the `location` column has been expanded out into latitude, longitude, street ID and street name.

In [7]:
crimes.head()

,category,location_type,location,context,outcome_status,persistent_id,id,location_subtype,month,latitude,longitude,street.id,street.name
0,anti-social-behaviour,Force,"{'latitude': '51.498404', 'street': {'id': 167...",,None,,113798885,,2023-10,51.498404,-0.160567,1673888,On or near Herbert Crescent
1,anti-social-behaviour,Force,"{'latitude': '51.517476', 'street': {'id': 167...",,None,,113798685,,2023-10,51.517476,-0.136579,1677079,On or near Supermarket
2,anti-social-behaviour,Force,"{'latitude': '51.511783', 'street': {'id': 167...",,None,,113798656,,2023-10,51.511783,-0.129778,1678199,On or near Nightclub
3,anti-social-behaviour,Force,"{'latitude': '51.507576', 'street': {'id': 167...",,None,,113798553,,2023-10,51.507576,-0.141776,1676403,On or near Nightclub
4,anti-social-behaviour,Force,"{'latitude': '51.516668', 'street': {'id': 167...",,None,,113795992,,2023-10,51.516668,-0.141210,1676419,On or near Margaret Street


As a final cleaning step, let's drop the original `location` column.

In [8]:
crimes.drop(columns=['location'],inplace=True)

---
## Exercise: Which crimes are most common?
    
Start off by performing some extra data cleaning steps:
    
* Remove the '-' in the `category` column
* Remove the phrase "on or near" in the `street.name` column

In [9]:
crimes['category'] = crimes['category'].str.replace('-',' ')
crimes['street.name'] = crimes['street.name'].str.replace('On or near ','')

Which streets were the most crimes recorded on?

In [10]:
crimes['street.name'].value_counts()

street.name
Nightclub                              775
Shopping Area                          354
Theatre/concert Hall                   304
Parking Area                           248
Further/higher Educational Building    208
                                      ... 
Lowndes Place                            1
Buckingham Gate                          1
Ebury Mews East                          1
Bryanston Mews West                      1
Montagu Place                            1
Name: count, Length: 506, dtype: int64

What kinds of crimes are the most common?

In [11]:
crimes['category'].value_counts()

category
theft from the person    1963
other theft              1748
violent crime             583
anti social behaviour     562
shoplifting               353
public order              245
vehicle crime             229
robbery                   223
burglary                  118
drugs                     116
criminal damage arson      97
bicycle theft              48
possession of weapons      21
other crime                20
Name: count, dtype: int64

---
## Exercise: How many crimes are solved? </font>
    
Now, let's combine our API and `pandas` skills to tackle a question.
    
We're now interested in how many crimes are **solved** in different parts of the UK. Can you use the **street level outcomes** section of the Police API (https://data.police.uk/docs/method/outcomes-at-location/) + your `pandas` skills to answer the question:
    
**If your bike is stolen in Oxford, how likely are you to get it back?**  (Either in Oxford itself, or for bonus points work out the surrounding area.)
    
There are several potential ways to answer this question - think about how you'll answer it before starting to code.
    
Remember to read the API documentation carefully, and experiment with demo API requests in your web browser **before** starting to write any code!

---
## <font color='green'> Sample solution: How many crimes are solved?
    
There are a few possible ways to answer this question.  In this sample solution we'll do it in 3 steps:
    
1. Determine how we can calculate the likelihood (to do this we'll need to understand what data is provided in the endpoint)
2. Get the data for one month and calculate the likelihood
3. Get all the data we can and work out the likelihood based on that

## 1. Determine how we can calculate the likelihood

The likelihood - or probability - of getting a bike back that has been stolen in Oxford can be calculated as:

$$\frac{The\; number\; of\; times\; bikes\; were\; returned}{Total\; number\; of\; bike\; thefts}$$

in any given time period.

By exploring the **Street-level outcomes** endpoint https://data.police.uk/docs/method/outcomes-at-location/ we can see that there are 28 categories of outcome for a crime:

- Awaiting court outcome
- Court result unavailable
- Court case unable to proceed
- Local resolution
- etc


Interestingly, there are no categories for property being returned to a victim, so there is no absolute way to tell from this data how many bikes are returned to people who have had them stolen.

However, we can make an assumption.

For the purposes of this exercise, we will assume that 'Local resolution' means that the bike was returned to the owner, and that no further police action was taken.  So our likelihood of getting a bike returned is now:

$$\frac{Number\; of\; Local\; Resolutions}{Total\; number\; of\; bike\; thefts}$$

We can get the **location of Oxford** by using Google Maps. For now we will just use one point (rather than multiple points, or the `poly` parameter) as the endpoint provides data within 1 mile radius of the point.


    latitude: 51.75157809244022
    longitude: -1.2565147281330131

## 2. Get the data for one month and calculate the likelihood

We can get the data for one month and calculate the likelihood using the `requests` library.

Let's start by getting all outcomes back for a given month.

In [12]:
lat= str(51.75157809244022)
lon= str(-1.2565147281330131)
date='2024-02'

street_url = 'https://data.police.uk/api/outcomes-at-location?date='+date+'&lat='+lat+'&lng='+lon

street_json = requests.get(street_url)

data = street_json.json()
# data

You get probably get hundreds of items back for the month you have selected

In [13]:
len(street_json.json())

406

The JSON is a list of dictionaries, with each dictionary representing an individual outcome of a crime.

Let's inspect the make up of one outcome to see how it is structured:

In [14]:
data[0]

{'category': {'code': 'unable-to-prosecute',
  'name': 'Unable to prosecute suspect'},
 'date': '2024-02',
 'person_id': None,
 'crime': {'category': 'other-theft',
  'location_type': 'Force',
  'location': {'latitude': '51.739715',
   'street': {'id': 1958857, 'name': 'On or near Summerfield'},
   'longitude': '-1.255024'},
  'context': '',
  'persistent_id': '81378fe033e0e2cf81d233de508c97f314382f4506eaf73b14c52a1311d752b2',
  'id': 116694885,
  'location_subtype': 'ROAD',
  'month': '2024-02'}}

There are 2 categories for each outcome: the category of the outcome e.g. `Unable to prosecute`; and the category of the crime e.g. `violent-crime`.

To find **bike thefts**, we need to look at the categories of crimes: https://data.police.uk/docs/method/outcomes-at-location/

(You can link to this page from the Street-level outcomes page)

In [15]:
requests.get('https://data.police.uk/api/crime-categories?date=2023-10').json()

[{'url': 'all-crime', 'name': 'All crime'},
 {'url': 'anti-social-behaviour', 'name': 'Anti-social behaviour'},
 {'url': 'bicycle-theft', 'name': 'Bicycle theft'},
 {'url': 'burglary', 'name': 'Burglary'},
 {'url': 'criminal-damage-arson', 'name': 'Criminal damage and arson'},
 {'url': 'drugs', 'name': 'Drugs'},
 {'url': 'other-theft', 'name': 'Other theft'},
 {'url': 'possession-of-weapons', 'name': 'Possession of weapons'},
 {'url': 'public-order', 'name': 'Public order'},
 {'url': 'robbery', 'name': 'Robbery'},
 {'url': 'shoplifting', 'name': 'Shoplifting'},
 {'url': 'theft-from-the-person', 'name': 'Theft from the person'},
 {'url': 'vehicle-crime', 'name': 'Vehicle crime'},
 {'url': 'violent-crime', 'name': 'Violence and sexual offences'},
 {'url': 'other-crime', 'name': 'Other crime'}]

So what we need to do to get **bike theft** data is to extract only those outcomes which have a crime category = `'bicycle-theft'`, and then count up how many of each category of the outcome we have.

We can use a `for` loop to do this.

In [16]:
data[0]['category']['code']

'unable-to-prosecute'

In [17]:
# create a dictionary to put the bike theft data into
bike_theft_data = {}

# iterate over each outcome, and if the crime category is 'bicycle-theft', then add the outcome category to the
# dictionary as a key, and give it a value of 1.  If the category already exists in the dictionary, then increment
# its value by 1.

for outcome in data:
    # print("for loop:", outcome)
    if outcome['crime']['category'] == 'bicycle-theft':
        if outcome['category']['code'] not in bike_theft_data:
            bike_theft_data[outcome['category']['code']] = 1
            # print(bike_theft_data)
            print("forloop_if:",bike_theft_data)
        else:
            bike_theft_data[outcome['category']['code']] += 1
            print(".....")
            print("forloop_else:",bike_theft_data)

bike_theft_data

forloop_if: {'no-further-action': 1}
.....
forloop_else: {'no-further-action': 2}
.....
forloop_else: {'no-further-action': 3}
.....
forloop_else: {'no-further-action': 4}
.....
forloop_else: {'no-further-action': 5}
forloop_if: {'no-further-action': 5, 'court-result-unavailable': 1}
.....
forloop_else: {'no-further-action': 6, 'court-result-unavailable': 1}
.....
forloop_else: {'no-further-action': 7, 'court-result-unavailable': 1}
.....
forloop_else: {'no-further-action': 8, 'court-result-unavailable': 1}
.....
forloop_else: {'no-further-action': 9, 'court-result-unavailable': 1}
.....
forloop_else: {'no-further-action': 10, 'court-result-unavailable': 1}
.....
forloop_else: {'no-further-action': 11, 'court-result-unavailable': 1}
.....
forloop_else: {'no-further-action': 12, 'court-result-unavailable': 1}
.....
forloop_else: {'no-further-action': 13, 'court-result-unavailable': 1}
.....
forloop_else: {'no-further-action': 14, 'court-result-unavailable': 1}
.....
forloop_else: {'no-f

{'no-further-action': 29,
 'court-result-unavailable': 1,
 'unable-to-prosecute': 1}

We now have the data we need to be able to calculate the likelihood given the data for one month.

In [18]:
# get the total of all outcomes relating to bike theft:

sum(bike_theft_data.values())

31

Since there are **no** Local Resolutions, the likelihood of getting your bike back given this data is 0.

But drawing that conclusion from only one month of data is not very sound as it may not be that representative of all months.  Let's get more data.

## 3. Get all the data we can and work out the likelihood based on that

In order to get data for many months, we can run the same code that we have already used, but use another `for` loop to get the data for each month.

First, we need to list out the months that we want to get data for.

The endpoint only provides the most recent 3 years of data.  Let's get all of it.

In [19]:
# create a list of 3 years of dates to gather outcomes for

dates = []
years = [2022, 2023, 2024, 2025]

for year in years:
    for month in range(1,13):
        dates.append(str(year)+'-'+str(month))

dates

['2022-1',
 '2022-2',
 '2022-3',
 '2022-4',
 '2022-5',
 '2022-6',
 '2022-7',
 '2022-8',
 '2022-9',
 '2022-10',
 '2022-11',
 '2022-12',
 '2023-1',
 '2023-2',
 '2023-3',
 '2023-4',
 '2023-5',
 '2023-6',
 '2023-7',
 '2023-8',
 '2023-9',
 '2023-10',
 '2023-11',
 '2023-12',
 '2024-1',
 '2024-2',
 '2024-3',
 '2024-4',
 '2024-5',
 '2024-6',
 '2024-7',
 '2024-8',
 '2024-9',
 '2024-10',
 '2024-11',
 '2024-12',
 '2025-1',
 '2025-2',
 '2025-3',
 '2025-4',
 '2025-5',
 '2025-6',
 '2025-7',
 '2025-8',
 '2025-9',
 '2025-10',
 '2025-11',
 '2025-12']

Let's create a list to contain all the data we get from the API.  (We will then use this to create a pandas DataFrame.)

In [20]:
all_data = []

Now we can tweak the code we used before (to get one month of data), so we can capture the month that we are getting data for, and we can get all the data for each month.

In [21]:
lat= str(51.75157809244022)
lon= str(-1.2565147281330131)


for date in dates:
    bike_theft_data = {}
    # add the month to the dictionary
    bike_theft_data['month'] = date

    # get bike theft data for each month
    street_url = 'https://data.police.uk/api/outcomes-at-location?date='+date+'&lat='+lat+'&lng='+lon
    street_json = requests.get(street_url)

    # some of the months will not be available (either they are more then 3 years ago, or have not happened yet)
    # these 2 lines make the code move to the next iteration if no data is returned, and allows the code to keep
    # running without error
    if street_json.status_code == 404:
        continue
    data = street_json.json()

    # use the same for loop as before to extract the bike theft outcome categories
    for outcome in data:
        if outcome['crime']['category'] == 'bicycle-theft':
            if outcome['category']['code'] not in bike_theft_data:
                bike_theft_data[outcome['category']['code']] = 1
            else:
                bike_theft_data[outcome['category']['code']] += 1
    # append the data for each month to the list of all data
    all_data.append(bike_theft_data)

In [22]:
all_data

[{'month': '2023-9', 'no-further-action': 32},
 {'month': '2023-10', 'no-further-action': 61},
 {'month': '2023-11',
  'no-further-action': 47,
  'court-result-unavailable': 2,
  'unable-to-prosecute': 2},
 {'month': '2023-12',
  'no-further-action': 31,
  'unable-to-prosecute': 2,
  'court-result-unavailable': 1},
 {'month': '2024-1', 'no-further-action': 27, 'unable-to-prosecute': 1},
 {'month': '2024-2',
  'no-further-action': 29,
  'court-result-unavailable': 1,
  'unable-to-prosecute': 1},
 {'month': '2024-3', 'no-further-action': 64, 'cautioned': 1},
 {'month': '2024-4', 'no-further-action': 44, 'court-result-unavailable': 2},
 {'month': '2024-5',
  'no-further-action': 36,
  'court-result-unavailable': 5,
  'unable-to-prosecute': 4},
 {'month': '2024-6',
  'no-further-action': 43,
  'court-result-unavailable': 6,
  'local-resolution': 1,
  'unable-to-prosecute': 3},
 {'month': '2024-7',
  'no-further-action': 56,
  'unable-to-prosecute': 4,
  'court-result-unavailable': 3},
 {'m

We can turn the data we have extracted into a pandas DataFrame.

In [23]:
df = pd.DataFrame(all_data)
df

,month,no-further-action,court-result-unavailable,unable-to-prosecute,cautioned,local-resolution,further-investigation-not-in-public-interest,sentenced-in-another-case,further-action-not-in-public-interest
0,2023-9,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-10,61,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-11,47,2.0,2.0,NaN,NaN,NaN,NaN,NaN
3,2023-12,31,1.0,2.0,NaN,NaN,NaN,NaN,NaN
4,2024-1,27,NaN,1.0,NaN,NaN,NaN,NaN,NaN
5,2024-2,29,1.0,1.0,NaN,NaN,NaN,NaN,NaN
6,2024-3,64,NaN,NaN,1.0,NaN,NaN,NaN,NaN
7,2024-4,44,2.0,NaN,NaN,NaN,NaN,NaN,NaN
8,2024-5,36,5.0,4.0,NaN,NaN,NaN,NaN,NaN
9,2024-6,43,6.0,3.0,NaN,1.0,NaN,NaN,NaN


We can see that there is now a **Local Resolution** category in some months.

To work out the likelihood of getting your bike back, we need to sum up all the **local-resolution**s, and sum up all the data.

In [24]:
# Add a 'Total' row that sums up each column
df.loc['Total'] = df.sum()

# Replace the value in the Month column
df.loc['Total', 'month'] = 'All'

df

,month,no-further-action,court-result-unavailable,unable-to-prosecute,cautioned,local-resolution,further-investigation-not-in-public-interest,sentenced-in-another-case,further-action-not-in-public-interest
0,2023-9,32,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023-10,61,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-11,47,2.0,2.0,NaN,NaN,NaN,NaN,NaN
3,2023-12,31,1.0,2.0,NaN,NaN,NaN,NaN,NaN
4,2024-1,27,NaN,1.0,NaN,NaN,NaN,NaN,NaN
5,2024-2,29,1.0,1.0,NaN,NaN,NaN,NaN,NaN
6,2024-3,64,NaN,NaN,1.0,NaN,NaN,NaN,NaN
7,2024-4,44,2.0,NaN,NaN,NaN,NaN,NaN,NaN
8,2024-5,36,5.0,4.0,NaN,NaN,NaN,NaN,NaN
9,2024-6,43,6.0,3.0,NaN,1.0,NaN,NaN,NaN


In [25]:
# get the total number of bike thefts that have had an outcome confirmed during the time period

total_bike_thefts = df.loc['Total'][1:].sum()
total_bike_thefts

1250.0

In [26]:
# Work out the likelihood

likelihood = df['local-resolution'].sum() / total_bike_thefts
print(f'The likelihood of getting your bike back is about {likelihood*100: .2f}%')

The likelihood of getting your bike back is about  0.32%


# Bonus

We can broaden our search to be more exact about the space we are seaching in.  To do this we use the `poly` parameter specified in the documentation.

In [27]:
all_data = []

In [28]:
# points around Oxford to create a broader area to search within

a = '51.787796997740735, -1.2837637088905183'
b = '51.79520329585142, -1.2715083485059857'
c = '51.771137196203334, -1.202505031426324'
d = '51.764523199909554, -1.177860757691506'
e = '51.72302745869018, -1.1829530687349075'
f = '51.711069927298624, -1.1979785311564242'
g = '51.70861571067208, -1.2211768387677524'
h = '51.717263310493465, -1.2351335441599336'
i = '51.72974878764573, -1.2786994807013587'
j = '51.75247029588256, -1.2812766516804153'

poly=a+':'+b+':'+c+':'+d+':'+e+':'+f+':'+g+':'+h+':'+i+':'+j
poly

'51.787796997740735, -1.2837637088905183:51.79520329585142, -1.2715083485059857:51.771137196203334, -1.202505031426324:51.764523199909554, -1.177860757691506:51.72302745869018, -1.1829530687349075:51.711069927298624, -1.1979785311564242:51.70861571067208, -1.2211768387677524:51.717263310493465, -1.2351335441599336:51.72974878764573, -1.2786994807013587:51.75247029588256, -1.2812766516804153'

In [29]:
for date in dates:
    bike_theft_data = {}
    # add the month to the dictionary
    bike_theft_data['month'] = date

    # get bike theft data for each month
    street_url = 'https://data.police.uk/api/outcomes-at-location?date='+date+'&poly='+poly
    street_json = requests.get(street_url)
    
    
    # as in the previous exercise, some of the months will not be available (either they are more then 3 years ago, or have not happened yet)
    # these 2 lines make the code move to the next iteration if no data is returned, and allows the code to keep
    # running without error
    if street_json.status_code == 404:
        continue
    data = street_json.json()

    # use the same for loop as before to extract the bike theft outcome categories
    for outcome in data:
        if outcome['crime']['category'] == 'bicycle-theft':
            if outcome['category']['code'] not in bike_theft_data:
                bike_theft_data[outcome['category']['code']] = 1
            else:
                bike_theft_data[outcome['category']['code']] += 1
    # append the data for each month to the list of all data
    all_data.append(bike_theft_data)

TypeError: string indices must be integers, not 'str'

In [ ]:
df = pd.DataFrame(all_data)
df

In [ ]:
# Add a 'Total' row that sums up each column
df.loc['Total'] = df.sum()

# Replace the value in the Month column
df.loc['Total', 'month'] = 'All'

df

In [ ]:
total_bike_thefts = df.loc['Total'][1:].sum()
likelihood = df['local-resolution'].sum() / total_bike_thefts
print(f'The likelihood of getting your bike back is about {likelihood*100: .2f}%')

In [ ]:
import json
import pandas as pd

# 1. An unbalanced JSON payload (varying structures, nested dictionaries, missing keys)
json_payload = """
[
    {"id": 1, "name": "Alice", "info": {"city": "New York", "age": 28}},
    {"id": 2, "name": "Bob"}, 
    {"id": 3, "name": "Charlie", "info": {"city": "Chicago"}, "skills": ["Python", "SQL"]}
]
"""

# 2. Parse the JSON string
data_list = json.loads(json_payload)

# 3. Loop through the unbalanced payload to safely extract data
rows = []
for item in data_list:
    # Use .get() to avoid KeyError if a key is missing
    info_dict = item.get("info", {}) 
    
    row_data = {
        "ID": item.get("id"),
        "Name": item.get("name"),
        # Extracting from nested dictionary with default fallbacks
        "City": info_dict.get("city", "Unknown"), 
        "Age": info_dict.get("age", None),
        # Handling a nested list by joining elements, or defaulting to empty string
        "Skills": ", ".join(item.get("skills", [])) if "skills" in item else "None"
    }
    rows.append(row_data)

# 4. Display in a DataFrame
df = pd.DataFrame(rows)
print(df)


   ID     Name      City   Age       Skills
0   1    Alice  New York  28.0         None
1   2      Bob   Unknown   NaN         None
2   3  Charlie   Chicago   NaN  Python, SQL


## Tip
- Learn about [Counter](https://realpython.com/python-counter/), the pythonic way to count objects (items)

---